# Retail Demand Forecasting: Data Preparation

In this notebook i have performed data ingestion, cleaning, merging, and feature engineering for a retail demand forecasting system.

---

## Load Data

Load the three CSV files from the `/data/` directory and inspect their structure.

In [139]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [140]:
# load train data
train = pd.read_csv('data/train_actual.csv')


print("TRAIN DATA")

print(f"Shape: {train.shape}")
print(f"\nColumns: {list(train.columns)}")
print(f"\nData Types:\n{train.dtypes}")
print(f"\nFirst 5 rows:")
display(train.head())

TRAIN DATA
Shape: (421570, 5)

Columns: ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']

Data Types:
Store             int64
Dept              int64
Date             object
Weekly_Sales    float64
IsHoliday          bool
dtype: object

First 5 rows:


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [141]:
# load features data
features = pd.read_csv('data/features_actual.csv')


print("FEATURES DATA")

print(f"Shape: {features.shape}")
print(f"\nColumns: {list(features.columns)}")
print(f"\nData Types:\n{features.dtypes}")
print(f"\nFirst 5 rows:")
display(features.head())

FEATURES DATA
Shape: (8190, 12)

Columns: ['Store', 'Date', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday']

Data Types:
Store             int64
Date             object
Temperature     float64
Fuel_Price      float64
MarkDown1       float64
MarkDown2       float64
MarkDown3       float64
MarkDown4       float64
MarkDown5       float64
CPI             float64
Unemployment    float64
IsHoliday          bool
dtype: object

First 5 rows:


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [142]:
# load stores data
stores = pd.read_csv('data/stores.csv')


print("STORES DATA")

print(f"Shape: {stores.shape}")
print(f"\nColumns: {list(stores.columns)}")
print(f"\nData Types:\n{stores.dtypes}")
print(f"\nFirst 5 rows:")
display(stores.head())

STORES DATA
Shape: (45, 3)

Columns: ['Store', 'Type', 'Size']

Data Types:
Store     int64
Type     object
Size      int64
dtype: object

First 5 rows:


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


---
## Data Cleaning

cleaning operations on each dataset to ensure data quality.

### clean train data

- Convert `Date` to datetime
- Remove rows where `Weekly_Sales` is negative
- Verify no duplicate rows

In [143]:
# convert Date to datetime
train['Date'] = pd.to_datetime(train['Date'])

print(f"Original train shape: {train.shape}")
print(f"Date column type: {train['Date'].dtype}")

Original train shape: (421570, 5)
Date column type: datetime64[ns]


In [144]:
# check for negative sales
negative_sales = train[train['Weekly_Sales'] < 0]
print(f"Number of rows with negative Weekly_Sales: {len(negative_sales)}")

if len(negative_sales) > 0:
    print(f"\nSample of negative sales:")
    display(negative_sales.head())
    
# remove negative sales
train = train[train['Weekly_Sales'] >= 0]
print(f"\nShape after removing negative sales: {train.shape}")

Number of rows with negative Weekly_Sales: 1285

Sample of negative sales:


,Store,Dept,Date,Weekly_Sales,IsHoliday
846,1,6,2012-08-10,-139.65,False
2384,1,18,2012-05-04,-1.27,False
6048,1,47,2010-02-19,-863.00,False
6049,1,47,2010-03-12,-698.00,False
6051,1,47,2010-10-08,-58.00,False



Shape after removing negative sales: (420285, 5)


In [145]:
# check for duplicates
duplicates = train.duplicated().sum()
print(f"Number of duplicate rows in train: {duplicates}")

if duplicates > 0:
    train = train.drop_duplicates()
    print(f"Shape after removing duplicates: {train.shape}")
else:
    print("No duplicates found")

Number of duplicate rows in train: 0
No duplicates found


### clean features data

- Convert `Date` to datetime
- Fill `MarkDown1-5` missing values with 0
- Forward-fill `CPI` and `Unemployment` per store

In [146]:
# convert Date to datetime
features['Date'] = pd.to_datetime(features['Date'])

print(f"Features shape: {features.shape}")
print(f"Date column type: {features['Date'].dtype}")
print(f"\nMissing values before cleaning:")
print(features.isnull().sum())

Features shape: (8190, 12)
Date column type: datetime64[ns]

Missing values before cleaning:
Store              0
Date               0
Temperature        0
Fuel_Price         0
MarkDown1       4158
MarkDown2       5269
MarkDown3       4577
MarkDown4       4726
MarkDown5       4140
CPI              585
Unemployment     585
IsHoliday          0
dtype: int64


In [147]:
# fill MarkDown columns with 0
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

for col in markdown_cols:
    if col in features.columns:
        features[col] = features[col].fillna(0)
        print(f"Filled {col} missing values with 0")

print(f"\nMissing values after filling markdowns:")
print(features.isnull().sum())

Filled MarkDown1 missing values with 0
Filled MarkDown2 missing values with 0
Filled MarkDown3 missing values with 0
Filled MarkDown4 missing values with 0
Filled MarkDown5 missing values with 0

Missing values after filling markdowns:
Store             0
Date              0
Temperature       0
Fuel_Price        0
MarkDown1         0
MarkDown2         0
MarkDown3         0
MarkDown4         0
MarkDown5         0
CPI             585
Unemployment    585
IsHoliday         0
dtype: int64


In [148]:
# forward-fill CPI and Unemployment per store
# sort by Store and Date first to ensure proper forward filling
features = features.sort_values(['Store', 'Date'])

# forward fill CPI and Unemployment within each store group
features['CPI'] = features.groupby('Store')['CPI'].fillna(method='ffill')
features['Unemployment'] = features.groupby('Store')['Unemployment'].fillna(method='ffill')

print("Forward-filled CPI and Unemployment per store")
print(f"\nMissing values after forward-fill:")
print(features.isnull().sum())


Forward-filled CPI and Unemployment per store

Missing values after forward-fill:
Store           0
Date            0
Temperature     0
Fuel_Price      0
MarkDown1       0
MarkDown2       0
MarkDown3       0
MarkDown4       0
MarkDown5       0
CPI             0
Unemployment    0
IsHoliday       0
dtype: int64


### clean stores data

- Check for duplicates
- Verify store size and type

In [149]:
# check for duplicates
duplicates = stores.duplicated().sum()
print(f"Number of duplicate rows in stores: {duplicates}")

if duplicates > 0:
    stores = stores.drop_duplicates()
    print(f"Shape after removing duplicates: {stores.shape}")
else:
    print("No duplicates found")

Number of duplicate rows in stores: 0
No duplicates found


In [150]:
# verify store attributes
print("Store Type distribution:")
print(stores['Type'].value_counts())

print(f"\nStore Size statistics:")
print(stores['Size'].describe())

# check for missing values
print(f"\nMissing values in stores:")
print(stores.isnull().sum())

# check for invalid store sizes (negative or zero)
invalid_sizes = stores[stores['Size'] <= 0]
if len(invalid_sizes) > 0:
    print(f"\nWarning: Found {len(invalid_sizes)} stores with invalid size")
    display(invalid_sizes)
else:
    print("\nAll store sizes are valid")

Store Type distribution:
Type
A    22
B    17
C     6
Name: count, dtype: int64

Store Size statistics:
count        45.000000
mean     130287.600000
std       63825.271991
min       34875.000000
25%       70713.000000
50%      126512.000000
75%      202307.000000
max      219622.000000
Name: Size, dtype: float64

Missing values in stores:
Store    0
Type     0
Size     0
dtype: int64

All store sizes are valid


---
## Merge Datasets

Merge train, features, and stores data into a single dataset.

In [151]:
# first merge: train + features on (Store, Date)
print(f"Train shape before merge: {train.shape}")
print(f"Features shape before merge: {features.shape}")

df = train.merge(features, on=['Store', 'Date'], how='left')

print(f"\nShape after merging train + features: {df.shape}")
print(f"Expected rows (same as train): {len(train)}")
print(f"Actual rows: {len(df)}")

Train shape before merge: (420285, 5)
Features shape before merge: (8190, 12)

Shape after merging train + features: (420285, 15)
Expected rows (same as train): 420285
Actual rows: 420285


In [152]:
# second merge: result + stores on Store
print(f"\nStores shape: {stores.shape}")

df = df.merge(stores, on='Store', how='left')

print(f"\nFinal shape after merging with stores: {df.shape}")
print(f"\nFinal columns: {list(df.columns)}")


Stores shape: (45, 3)

Final shape after merging with stores: (420285, 17)

Final columns: ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday_x', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday_y', 'Type', 'Size']


In [153]:
# verify no unexpected data loss
print("Data integrity check:")
print(f"Original train rows: {len(train)}")
print(f"Final merged rows: {len(df)}")
print(f"Difference: {len(df) - len(train)}")

if len(df) == len(train):
    print("\nAll train records preserved")
elif len(df) < len(train):
    print(f"\nLost {len(train) - len(df)} records")
else:
    print(f"\nGained {len(df) - len(train)} records (possible duplication)")

Data integrity check:
Original train rows: 420285
Final merged rows: 420285
Difference: 0

All train records preserved


---
## Create Time Features

Extract time-based features from the Date column.

In [154]:
# create time features
df['Year'] = df['Date'].dt.year
df['Week'] = df['Date'].dt.isocalendar().week
df['Month'] = df['Date'].dt.month

print("Created time features: Year, Week, Month")
print(f"\nTime features summary:")
print(df[['Date', 'Year', 'Week', 'Month']].head(10))

print(f"\nYear range: {df['Year'].min()} - {df['Year'].max()}")
print(f"Week range: {df['Week'].min()} - {df['Week'].max()}")
print(f"Month range: {df['Month'].min()} - {df['Month'].max()}")

Created time features: Year, Week, Month

Time features summary:
        Date  Year  Week  Month
0 2010-02-05  2010     5      2
1 2010-02-12  2010     6      2
2 2010-02-19  2010     7      2
3 2010-02-26  2010     8      2
4 2010-03-05  2010     9      3
5 2010-03-12  2010    10      3
6 2010-03-19  2010    11      3
7 2010-03-26  2010    12      3
8 2010-04-02  2010    13      4
9 2010-04-09  2010    14      4

Year range: 2010 - 2012
Week range: 1 - 52
Month range: 1 - 12


---
## Create Sales History Features

Create lag and rolling features for each (Store, Dept) group.

**Important:** These features use `.shift()` to ensure no data leakage from future observations.

In [155]:
# sort by Store, Dept, and Date to ensure proper chronological order
df = df.sort_values(['Store', 'Dept', 'Date'])
print(f"Shape: {df.shape}")

Shape: (420285, 20)


In [156]:
# create lag features
# using shift() ensures we only use past data (no leakage)
df['sales_lag_1'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1)
df['sales_lag_2'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(2)
df['sales_lag_4'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(4)
df['sales_lag_8'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(8)

print("Created lag features: sales_lag_1, sales_lag_2, sales_lag_4, sales_lag_8")

# show sample to verify lags are correct
print("\nSample to verify lag features (first store-dept combination):")
sample = df[df['Store'] == df['Store'].iloc[0]]
sample = sample[sample['Dept'] == sample['Dept'].iloc[0]]
display(sample[['Date', 'Store', 'Dept', 'Weekly_Sales', 'sales_lag_1', 'sales_lag_2']].head(10))

Created lag features: sales_lag_1, sales_lag_2, sales_lag_4, sales_lag_8

Sample to verify lag features (first store-dept combination):


,Date,Store,Dept,Weekly_Sales,sales_lag_1,sales_lag_2
0,2010-02-05,1,1,24924.50,NaN,NaN
1,2010-02-12,1,1,46039.49,24924.50,NaN
2,2010-02-19,1,1,41595.55,46039.49,24924.50
3,2010-02-26,1,1,19403.54,41595.55,46039.49
4,2010-03-05,1,1,21827.90,19403.54,41595.55
5,2010-03-12,1,1,21043.39,21827.90,19403.54
6,2010-03-19,1,1,22136.64,21043.39,21827.90
7,2010-03-26,1,1,26229.21,22136.64,21043.39
8,2010-04-02,1,1,57258.43,26229.21,22136.64
9,2010-04-09,1,1,42960.91,57258.43,26229.21


In [157]:
# create rolling mean features
# using shift(1) before rolling ensures we only use past data
# rolling mean of last 4 weeks (excluding current week)
df['rolling_mean_4'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=1).mean()
)

# rolling mean of last 8 weeks (excluding current week)
df['rolling_mean_8'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].transform(
    lambda x: x.shift(1).rolling(window=8, min_periods=1).mean()
)


# show sample to verify rolling features
print("\nSample to verify rolling features:")
sample = df[df['Store'] == df['Store'].iloc[0]]
sample = sample[sample['Dept'] == sample['Dept'].iloc[0]]
display(sample[['Date', 'Weekly_Sales', 'sales_lag_1', 'rolling_mean_4', 'rolling_mean_8']].head(15))


Sample to verify rolling features:


,Date,Weekly_Sales,sales_lag_1,rolling_mean_4,rolling_mean_8
0,2010-02-05,24924.50,NaN,NaN,NaN
1,2010-02-12,46039.49,24924.50,24924.500000,24924.500000
2,2010-02-19,41595.55,46039.49,35481.995000,35481.995000
3,2010-02-26,19403.54,41595.55,37519.846667,37519.846667
4,2010-03-05,21827.90,19403.54,32990.770000,32990.770000
5,2010-03-12,21043.39,21827.90,32216.620000,30758.196000
6,2010-03-19,22136.64,21043.39,25967.595000,29139.061667
7,2010-03-26,26229.21,22136.64,21102.867500,28138.715714
8,2010-04-02,57258.43,26229.21,22809.285000,27900.027500
9,2010-04-09,42960.91,57258.43,31666.917500,31941.768750


In [158]:
# check for null values in new features
print("Missing values in new features:")

lag_rolling_cols = ['sales_lag_1', 'sales_lag_2', 'sales_lag_4', 'sales_lag_8', 
                    'rolling_mean_4', 'rolling_mean_8']
for col in lag_rolling_cols:
    null_count = df[col].isnull().sum()
    null_pct = (null_count / len(df)) * 100
    print(f"{col:20s}: {null_count:6d} ({null_pct:5.2f}%)")

print("\nFirst few rows per Store-Dept will have null lag values")

Missing values in new features:
sales_lag_1         :   3323 ( 0.79%)
sales_lag_2         :   6606 ( 1.57%)
sales_lag_4         :  13097 ( 3.12%)
sales_lag_8         :  25859 ( 6.15%)
rolling_mean_4      :   3323 ( 0.79%)
rolling_mean_8      :   3323 ( 0.79%)

First few rows per Store-Dept will have null lag values


---
## Normalize Store Size

Create a normalized sales metric: sales per square foot.

In [159]:
# create sales per square foot
df['sales_per_sqft'] = df['Weekly_Sales'] / df['Size']

print("Created sales_per_sqft = Weekly_Sales / Size")
print(f"\nSales per sqft statistics:")
print(df['sales_per_sqft'].describe())

print(f"\nSample rows with new feature:")
display(df[['Store', 'Dept', 'Weekly_Sales', 'Size', 'sales_per_sqft']].head(10))

Created sales_per_sqft = Weekly_Sales / Size

Sales per sqft statistics:
count    420285.000000
mean          0.130006
std           0.204320
min           0.000000
25%           0.017591
50%           0.059204
75%           0.161369
max           6.267013
Name: sales_per_sqft, dtype: float64

Sample rows with new feature:


,Store,Dept,Weekly_Sales,Size,sales_per_sqft
0,1,1,24924.50,151315,0.164719
1,1,1,46039.49,151315,0.304263
2,1,1,41595.55,151315,0.274894
3,1,1,19403.54,151315,0.128233
4,1,1,21827.90,151315,0.144255
5,1,1,21043.39,151315,0.139070
6,1,1,22136.64,151315,0.146295
7,1,1,26229.21,151315,0.173342
8,1,1,57258.43,151315,0.378406
9,1,1,42960.91,151315,0.283917


---
## Final Feature Table

Select the final features and prepare the dataset for modeling.

In [160]:
# define final feature columns
final_columns = [
    # identifiers
    'Store', 'Dept', 'Date',
    
    # target
    'Weekly_Sales',
    
    # time features
    'Year', 'Week', 'Month',
    
    # lag features
    'sales_lag_1', 'sales_lag_2', 'sales_lag_4', 'sales_lag_8',
    
    # rolling features
    'rolling_mean_4', 'rolling_mean_8',
    
    # store attributes
    'Type', 'Size',
    
    # normalized metric
    'sales_per_sqft',
    
    # economic & promotional features
    'IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5'
]

# select only columns that exist in the dataframe
available_columns = [col for col in final_columns if col in df.columns]
missing_columns = [col for col in final_columns if col not in df.columns]

print("Feature selection:")

print(f"Requested columns: {len(final_columns)}")
print(f"Available columns: {len(available_columns)}")

if missing_columns:
    print(f"\nMissing columns: {missing_columns}")
else:
    print("\nAll requested columns are available")

# select final features
df_final = df[available_columns].copy()

print(f"\nFinal dataset shape before dropping nulls: {df_final.shape}")

Feature selection:
Requested columns: 26
Available columns: 25

Missing columns: ['IsHoliday']

Final dataset shape before dropping nulls: (420285, 25)


In [161]:
# drop rows where lag features are null
print(f"Shape before: {df_final.shape}")

# drop rows where any lag feature is null
lag_cols = ['sales_lag_1', 'sales_lag_2', 'sales_lag_4', 'sales_lag_8']
df_final = df_final.dropna(subset=lag_cols)

print(f"Shape after: {df_final.shape}")
print(f"Rows removed: {df.shape[0] - df_final.shape[0]}")

Shape before: (420285, 25)
Shape after: (394426, 25)
Rows removed: 25859


In [162]:
# display final dataset summary

print("FINAL FEATURE TABLE SUMMARY")

print(f"\nShape: {df_final.shape}")
print(f"\nColumns ({len(df_final.columns)}):")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nData types:")
print(df_final.dtypes)

print(f"\nFirst 10 rows:")
display(df_final.head(10))

print(f"\nNumerical features summary:")
display(df_final.describe())

FINAL FEATURE TABLE SUMMARY

Shape: (394426, 25)

Columns (25):
   1. Store
   2. Dept
   3. Date
   4. Weekly_Sales
   5. Year
   6. Week
   7. Month
   8. sales_lag_1
   9. sales_lag_2
  10. sales_lag_4
  11. sales_lag_8
  12. rolling_mean_4
  13. rolling_mean_8
  14. Type
  15. Size
  16. sales_per_sqft
  17. Temperature
  18. Fuel_Price
  19. CPI
  20. Unemployment
  21. MarkDown1
  22. MarkDown2
  23. MarkDown3
  24. MarkDown4
  25. MarkDown5

Data types:
Store                      int64
Dept                       int64
Date              datetime64[ns]
Weekly_Sales             float64
Year                       int32
Week                      UInt32
Month                      int32
sales_lag_1              float64
sales_lag_2              float64
sales_lag_4              float64
sales_lag_8              float64
rolling_mean_4           float64
rolling_mean_8           float64
Type                      object
Size                       int64
sales_per_sqft           float64
Tempera

,Store,Dept,Date,Weekly_Sales,Year,Week,Month,sales_lag_1,sales_lag_2,sales_lag_4,sales_lag_8,rolling_mean_4,rolling_mean_8,Type,Size,sales_per_sqft,Temperature,Fuel_Price,CPI,Unemployment,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5
8,1,1,2010-04-02,57258.43,2010,13,4,26229.21,22136.64,21827.90,24924.50,22809.2850,27900.02750,A,151315,0.378406,62.27,2.719,210.820450,7.808,0.0,0.0,0.0,0.0,0.0
9,1,1,2010-04-09,42960.91,2010,14,4,57258.43,26229.21,21043.39,46039.49,31666.9175,31941.76875,A,151315,0.283917,65.86,2.770,210.622857,7.808,0.0,0.0,0.0,0.0,0.0
10,1,1,2010-04-16,17596.96,2010,15,4,42960.91,57258.43,22136.64,41595.55,37146.2975,31556.94625,A,151315,0.116294,66.32,2.808,210.488700,7.808,0.0,0.0,0.0,0.0,0.0
11,1,1,2010-04-23,16145.35,2010,16,4,17596.96,42960.91,26229.21,19403.54,36011.3775,28557.12250,A,151315,0.106700,64.84,2.795,210.439123,7.808,0.0,0.0,0.0,0.0,0.0
12,1,1,2010-04-30,16555.11,2010,17,4,16145.35,17596.96,57258.43,21827.90,33490.4125,28149.84875,A,151315,0.109408,67.41,2.780,210.389546,7.808,0.0,0.0,0.0,0.0,0.0
13,1,1,2010-05-07,17413.94,2010,18,5,16555.11,16145.35,42960.91,21043.39,23314.5825,27490.75000,A,151315,0.115084,72.55,2.835,210.339968,7.808,0.0,0.0,0.0,0.0,0.0
14,1,1,2010-05-14,18926.74,2010,19,5,17413.94,16555.11,17596.96,22136.64,16927.8400,27037.06875,A,151315,0.125082,74.78,2.854,210.337426,7.808,0.0,0.0,0.0,0.0,0.0
15,1,1,2010-05-21,14773.04,2010,20,5,18926.74,17413.94,16145.35,26229.21,17260.2850,26635.83125,A,151315,0.097631,76.44,2.826,210.617093,7.808,0.0,0.0,0.0,0.0,0.0
16,1,1,2010-05-28,15580.43,2010,21,5,14773.04,18926.74,16555.11,57258.43,16917.2075,25203.81000,A,151315,0.102967,80.44,2.759,210.896761,7.808,0.0,0.0,0.0,0.0,0.0
17,1,1,2010-06-04,17558.09,2010,22,6,15580.43,14773.04,17413.94,42960.91,16673.5375,19994.06000,A,151315,0.116037,80.69,2.705,211.176428,7.808,0.0,0.0,0.0,0.0,0.0



Numerical features summary:


,Store,Dept,Date,Weekly_Sales,Year,Week,Month,sales_lag_1,sales_lag_2,sales_lag_4,sales_lag_8,rolling_mean_4,rolling_mean_8,Size,sales_per_sqft,Temperature,Fuel_Price,CPI,Unemployment,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5
count,394426.000000,394426.000000,394426,394426.000000,394426.000000,394426.0,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000,394426.000000
mean,22.180219,44.184658,2011-07-17 06:20:08.622149376,16136.192472,2011.028013,26.866758,6.686276,16132.769739,16132.531902,16130.560373,16155.459666,16132.010222,16138.485856,136879.610728,0.130841,61.208201,3.398626,171.433974,7.921625,2753.255440,933.553497,492.139582,1151.046020,1765.831725
min,1.000000,1.000000,2010-04-02 00:00:00,0.000000,2010.000000,1.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.020000,0.017500,34875.000000,0.000000,-2.060000,2.513000,126.064000,3.879000,0.000000,-265.760000,-29.100000,0.000000,0.000000
25%,11.000000,18.000000,2010-11-26 00:00:00,2185.270000,2010.000000,16.0,4.000000,2185.572500,2186.500000,2183.957500,2188.725000,2250.391250,2293.510000,93638.000000,0.018041,48.430000,3.001000,132.473333,6.877000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,22.000000,37.000000,2011-07-15 00:00:00,7738.915000,2011.000000,27.0,7.000000,7737.590000,7740.870000,7741.130000,7767.115000,7853.972500,7914.766875,140167.000000,0.059891,63.630000,3.501000,182.544590,7.852000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,33.000000,74.000000,2012-03-09 00:00:00,20416.587500,2012.000000,38.0,9.000000,20409.602500,20410.207500,20406.490000,20442.422500,20500.546250,20589.791875,202505.000000,0.162825,75.090000,3.749000,213.023622,8.549000,3296.230000,7.640000,6.000000,542.890000,2399.330000
max,45.000000,99.000000,2012-10-26 00:00:00,693099.360000,2012.000000,52.0,12.000000,693099.360000,693099.360000,693099.360000,693099.360000,339472.757500,275433.013750,219622.000000,6.267013,100.140000,4.468000,227.232807,14.313000,88646.760000,104519.540000,141630.610000,67474.850000,108519.280000
std,12.784542,30.532154,NaN,22789.876408,0.783333,13.893236,3.186078,22784.205198,22780.343441,22779.366341,22813.978647,22262.480378,22088.657700,60941.499890,0.204619,18.162549,0.443945,39.215298,1.863758,6203.292477,5227.590563,5667.951544,4002.806235,4313.115205


---
## Save Output

Save the final feature table to `/data/processed/walmart_features.csv`

In [163]:
# create output directory if it doesn't exist
output_dir = 'data/processed'
os.makedirs(output_dir, exist_ok=True)

In [164]:
# save to CSV
output_file = os.path.join(output_dir, 'walmart_features.csv')
df_final.to_csv(output_file, index=False)

print(f"File: {output_file}")
print(f"Rows: {len(df_final):,}")
print(f"Columns: {len(df_final.columns)}")


File: data/processed\walmart_features.csv
Rows: 394,426
Columns: 25
